# NASA GIBS satellite tiles with an administrative overlay

This notebook combines NASA Global Imagery Browse Services-style web tiles with a real district boundary overlay. It is designed to build intuition for remote sensing layers before downloading large rasters. <iframe width="560" height="315" src="https://www.youtube.com/embed/X16cfGPL2wA" title="NASA Worldview" frameborder="0" allowfullscreen></iframe>

Data: Montreal districts are local GeoJSON; the satellite base layer is requested from NASA GIBS WMTS at display time.

**Reflection questions:** What can satellite imagery show that vector boundaries hide? What cannot be inferred safely from a natural-color image? Which dates or seasons would matter for your question?

In [ ]:
# Pyodide/JupyterLite bootstrap: install only pure-Python packages used in this notebook.
import sys, importlib
try:
    import micropip
except Exception:
    micropip = None

async def ensure_packages(packages):
    for pkg, import_name in packages:
        try:
            importlib.import_module(import_name)
        except Exception:
            if micropip is None:
                raise RuntimeError(f'{pkg} is not installed and micropip is unavailable.')
            await micropip.install(pkg)

await ensure_packages([('pandas','pandas'), ('folium','folium'), ('branca','branca'), ('plotly','plotly')])


In [ ]:
from pathlib import Path
import json, math, statistics
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap, TimestampedGeoJson, MiniMap, Fullscreen, MeasureControl

DATA = Path('../data')

def load_json(name):
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def load_csv(name):
    return pd.read_csv(DATA / name)

def add_standard_controls(m):
    MiniMap(toggle_display=True).add_to(m)
    Fullscreen().add_to(m)
    MeasureControl(primary_length_unit='kilometers').add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

def color_scale(values, colors=('green','orange','red')):
    vals = list(values)
    lo, hi = min(vals), max(vals)
    def pick(v):
        if hi == lo:
            return colors[1]
        t = (v - lo) / (hi - lo)
        return colors[0] if t < .33 else colors[1] if t < .66 else colors[2]
    return pick


In [ ]:
districts = load_json('montreal_districts.geojson')
m = folium.Map(location=[45.52, -73.60], zoom_start=10, tiles=None)
folium.TileLayer('OpenStreetMap', name='OpenStreetMap').add_to(m)
# NASA GIBS layer URL pattern; date can be changed for time-specific exploration.
gibs = 'https://gibs.earthdata.nasa.gov/wmts/epsg3857/best/MODIS_Terra_CorrectedReflectance_TrueColor/default/2024-07-01/GoogleMapsCompatible_Level9/{z}/{y}/{x}.jpg'
folium.TileLayer(tiles=gibs, attr='NASA GIBS', name='NASA MODIS Terra True Color', overlay=False, control=True).add_to(m)
folium.GeoJson(districts, name='Montreal electoral districts', style_function=lambda f: {'fillOpacity':0, 'weight':1.5}).add_to(m)
add_standard_controls(m)
m